# Урок 23. Переобучение и честный выбор модели

**Дата:** 25.02.2027 · **60 минут** · 8–9 класс  
**Модуль 4.** Машинное обучение (уроки 17–25)  
**Опирается на:** уроки 19–22 (деревья, kNN, train / test, метрики), урок 13 (графики).

Мы уже несколько раз видели странное: глубокое дерево на train почти безошибочно, а на test — хуже простого. kNN с `k = 1` «угадывает» всех в train. Сегодня разберёмся с этим явлением — **переобучением** — и научимся **честно** выбирать модель и её настройки, не подглядывая в test.

## Цели урока

К концу занятия ты сможешь:

1. Объяснять **недообучение** и **переобучение** и узнавать их по цифрам train и test.
2. Строить график «качество от сложности модели».
3. Называть причины переобучения и способы с ним бороться.
4. Выбирать настройки модели по **валидационной** выборке, а test использовать один раз.
5. Проверять устойчивость результата на нескольких разбиениях.

## План на 60 минут

| Минуты | Что делаем |
|-------:|------------|
| 0–8 | Три ученика: недоучил, понял, зазубрил |
| 8–25 | Эксперимент: сложность модели и качество |
| 25–40 | Валидационная выборка и честный выбор |
| 40–55 | Практика |
| 55–60 | Итог, подготовка к СР №3 |

## 1. Три ученика

Перед контрольной три ученика готовятся по сборнику задач:

| Ученик | Как готовился | На задачах из сборника | На контрольной (новые задачи) | В ML это… |
|--------|---------------|:----------------------:|:-----------------------------:|-----------|
| Лентяй | пролистал | плохо | плохо | **недообучение** |
| Зубрила | выучил ответы наизусть | идеально | плохо | **переобучение** |
| Понимающий | разобрался в методе | хорошо | хорошо | **хорошая модель** |

- **Недообучение** — модель слишком простая, не уловила даже основную закономерность: плохо и на train, и на test.
- **Переобучение** — модель слишком сложная, выучила вместе с закономерностью и **случайный шум** обучающих данных: отлично на train, хуже на test.

**Как узнать по цифрам:** большой разрыв «train намного лучше test» — признак переобучения. Оба низкие — недообучение.

## 2. Данные: как в жизни — шумные и с лишними признаками

В реальных данных много шума (на балл влияет масса случайностей) и есть столбцы, которые к задаче не относятся. Сделаем данные похожими на настоящие: шума больше, чем раньше, и добавим два бесполезных признака — шаги в день и рост.

In [ ]:
import random
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

def make_students(n, seed):
    """Учебные данные: больше шума и два бесполезных признака."""
    random.seed(seed)
    rows = []
    for i in range(n):
        hours = random.randint(1, 14)
        attendance = min(100, max(20, round(50 + 3 * hours + random.gauss(0, 10))))
        score = min(100, max(0, round(20 + 4 * hours + 0.3 * attendance + random.gauss(0, 14))))
        rows.append({
            "часы": hours,
            "посещаемость": attendance,
            "сон": round(random.uniform(6, 9.5), 1),
            "шаги": random.randint(3000, 15000),
            "рост": random.randint(150, 185),
            "балл": score,
        })
    return pd.DataFrame(rows)

df = make_students(200, seed=23)
df["сдал"] = df["балл"] >= 60
X = df[["часы", "посещаемость", "сон", "шаги", "рост"]]
y = df["сдал"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print("train:", len(X_train), "| test:", len(X_test), "| доля сдавших:", round(y.mean(), 2))

## 3. Эксперимент: сложность модели и качество

Сложность дерева — его глубина. Обучим деревья разной глубины и запишем accuracy на train и на test в таблицу.

In [ ]:
rows = []
for depth in [1, 2, 3, 4, 6, 8, 12, 20]:
    model = DecisionTreeClassifier(max_depth=depth, random_state=0)
    model.fit(X_train, y_train)
    rows.append({
        "глубина": depth,
        "train": accuracy_score(y_train, model.predict(X_train)),
        "test": accuracy_score(y_test, model.predict(X_test)),
    })
results = pd.DataFrame(rows)
results["разрыв"] = results["train"] - results["test"]
results.round(2)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(results["глубина"], results["train"], marker="o", label="train")
plt.plot(results["глубина"], results["test"], marker="o", label="test")
plt.title("Чем сложнее дерево, тем больше разрыв")
plt.xlabel("Глубина дерева (сложность модели)")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Как читать график:**

- train растёт до 1.0 — глубокое дерево запоминает каждого ученика;
- test **не растёт**, а даже падает — запомненные подробности шумные и на новых учениках только мешают;
- разрыв между линиями растёт вместе с глубиной — это и есть переобучение.

Хорошая модель — не та, где train максимален, а та, где **test** высокий и разрыв небольшой.

## 4. Почему модели переобучаются

| Причина | Что происходит | Что помогает |
|---------|----------------|--------------|
| Модель слишком сложная | подстраивается под каждую точку | ограничить сложность: `max_depth`, больше `k` у kNN |
| Мало данных | шум легко принять за закономерность | собрать больше данных |
| Много лишних признаков | модель находит случайные «закономерности» в бесполезных столбцах | убрать лишние признаки (разведка, урок 12) |
| Шумные ответы | модель заучивает ошибки разметки | почистить данные (урок 11) |

Посмотри, какие признаки использует самое глубокое дерево:

In [ ]:
deep = DecisionTreeClassifier(max_depth=20, random_state=0)
deep.fit(X_train, y_train)
print(pd.Series(deep.feature_importances_, index=X.columns).round(2).sort_values(ascending=False))

Больше трети решений глубокого дерева опирается на **сон, шаги и рост** — признаки, которые с результатом **никак не связаны** (мы так задали данные). Дерево нашло в них случайные совпадения в train. Это классический симптом переобучения.

## 5. Честный выбор настроек: валидационная выборка

Мы хотим выбрать лучшую глубину. Соблазн: выбрать ту, где лучше всего на test. Но тогда test перестаёт быть «новыми данными»: мы **подобрали** под него настройку, и итоговая оценка будет завышенной — как будто ученик подсмотрел варианты контрольной.

Решение — делить данные на **три** части:

| Часть | Зачем | Аналогия |
|-------|-------|----------|
| **train** | обучаем модели | разбор задач в классе |
| **validation** (валидационная) | сравниваем модели и выбираем настройки | пробная контрольная |
| **test** | **один раз** проверяем выбранную модель | настоящая контрольная |

Сделаем это двумя вызовами `train_test_split`: сначала откладываем test, потом из оставшегося — validation.

In [ ]:
X_rest, X_test, y_rest, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_rest, y_rest, test_size=0.25, random_state=42, stratify=y_rest
)
print("train:", len(X_train), "| validation:", len(X_val), "| test:", len(X_test))

In [ ]:
best_depth = None
best_val = 0
for depth in [1, 2, 3, 4, 6, 8, 12]:
    model = DecisionTreeClassifier(max_depth=depth, random_state=0)
    model.fit(X_train, y_train)
    val_acc = accuracy_score(y_val, model.predict(X_val))
    print(f"глубина {depth:2}: validation {val_acc:.2f}")
    if val_acc > best_val:
        best_val = val_acc
        best_depth = depth

print("Выбрали глубину:", best_depth)

Глубину выбрали по validation — test мы ещё **ни разу не трогали**. Теперь — единственная проверка на test:

In [ ]:
final = DecisionTreeClassifier(max_depth=best_depth, random_state=0)
final.fit(X_train, y_train)
print("Итоговая accuracy на test:", round(accuracy_score(y_test, final.predict(X_test)), 2))

Именно это число честно сообщают в отчёте. Если после этого «подкрутить» модель и снова посмотреть на test — он превращается во вторую validation, и честной оценки больше нет.

## 6. Одно разбиение — это лотерея

На уроке 19 мы видели: при другом `random_state` accuracy на test заметно меняется. В маленьком test пара «трудных» учеников может сдвинуть результат. Чтобы сравнение было надёжным, повторяют эксперимент на **нескольких** разбиениях и смотрят на **среднее**.

In [ ]:
for depth in [2, 4, 12]:
    scores = []
    for seed in range(10):
        Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)
        model = DecisionTreeClassifier(max_depth=depth, random_state=0)
        model.fit(Xa, ya)
        scores.append(accuracy_score(yb, model.predict(Xb)))
    print(f"глубина {depth:2}: среднее {sum(scores) / len(scores):.3f}, "
          f"от {min(scores):.2f} до {max(scores):.2f}")

Видно две вещи: по среднему простые деревья стабильно не хуже сложных, а разброс между разбиениями — несколько процентов. Поэтому разница в 1–2% между моделями на одном разбиении **ничего не доказывает**. Повторение проверки на разных разбиениях — основа метода, который называют **перекрёстной проверкой** (*cross-validation*).

## 7. Как выбрать модель

Качество на validation — главный, но не единственный критерий:

| Критерий | Вопрос |
|----------|--------|
| Качество | Лучше ли baseline? Хорошая ли главная метрика (урок 21)? |
| Разрыв train / test | Нет ли переобучения? |
| Устойчивость | Похожий результат на разных разбиениях? |
| Понятность | Можно ли объяснить решения людям? Дерево глубины 3 можно распечатать и прочитать |
| Простота | Если две модели почти одинаковы — бери **более простую** |

Последнее правило называют **бритвой Оккама**: из двух одинаково хороших объяснений выбирай более простое.

## 8. Практика

### Практика 1. Выбери k для kNN (10 минут)

Используя `X_train`, `X_val`, `y_train`, `y_val` из раздела 5, подбери `k` для kNN из списка `[1, 3, 5, 9, 15, 31]` по validation. Не забудь масштабирование по train (урок 20). Потом **один раз** проверь лучший `k` на test.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
# Практика 1

<details>
<summary>Решение</summary>

```python
max_train = X_train.max()
best_k = None
best_val_k = 0
for k in [1, 3, 5, 9, 15, 31]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train / max_train, y_train)
    train_acc = accuracy_score(y_train, knn.predict(X_train / max_train))
    val_acc = accuracy_score(y_val, knn.predict(X_val / max_train))
    print(f"k = {k:2}: train {train_acc:.2f}, validation {val_acc:.2f}")
    if val_acc > best_val_k:
        best_val_k = val_acc
        best_k = k

knn_final = KNeighborsClassifier(n_neighbors=best_k)
knn_final.fit(X_train / max_train, y_train)
print("Лучший k:", best_k, "| test:", round(accuracy_score(y_test, knn_final.predict(X_test / max_train)), 2))
```
При `k = 1` на train всегда 1.0 — переобучение. При очень большом `k` модель становится слишком «усреднённой» — недообучение.
</details>

### Практика 2. Убери лишнее (8 минут)

Обучи глубокое дерево (`max_depth=12`) только на признаках `часы` и `посещаемость`. Сравни train и validation с деревом на всех пяти признаках. Что изменилось, а что нет? Почему?

In [ ]:
# Практика 2

<details>
<summary>Решение</summary>

```python
for cols in [["часы", "посещаемость", "сон", "шаги", "рост"], ["часы", "посещаемость"]]:
    model = DecisionTreeClassifier(max_depth=12, random_state=0)
    model.fit(X_train[cols], y_train)
    tr = accuracy_score(y_train, model.predict(X_train[cols]))
    va = accuracy_score(y_val, model.predict(X_val[cols]))
    print(len(cols), "признаков: train", round(tr, 2), "| validation", round(va, 2))
```
Без бесполезных признаков validation становится чуть лучше, но разрыв не исчезает: шум в самих ответах глубокое дерево всё равно запоминает. Помогает сочетание мер — убрать лишние признаки **и** ограничить глубину.
</details>

### Практика 3. Отчёт о выборе модели (5 минут)

Напиши 3–4 предложения: какую модель и с какими настройками ты выбрал бы для этой задачи, по какой выборке выбирал, какое итоговое качество на test и почему не взял самое глубокое дерево.

…

## Работа с ИИ-помощником

- «Объясни разницу между validation и test на примере подготовки к экзамену.»
- «У меня train 1.0, validation 0.78. Что это значит и что попробовать?»
- Проверяй советы помощника экспериментом: «попробуй увеличить глубину» — сначала посмотри, что станет с validation.

## Мини-глоссарий урока 23

| Термин | Смысл |
|--------|-------|
| **Недообучение** | модель слишком простая: плохо и на train, и на test |
| **Переобучение** | модель запомнила шум: отлично на train, хуже на новых данных |
| **Сложность модели** | глубина дерева, малое `k` у kNN и т. п. |
| **Validation** | выборка для сравнения моделей и выбора настроек |
| **Test** | выборка для одной итоговой проверки |
| **Перекрёстная проверка** | повтор оценки на нескольких разбиениях |
| **Бритва Оккама** | из одинаково хороших моделей выбирают более простую |

## Итог урока

1. Переобучение видно по разрыву: train намного лучше, чем test.
2. Настройки выбирают по **validation**; test открывают **один раз** в конце.
3. Одно разбиение — лотерея: проверяй на нескольких и выбирай более простую из равных моделей.

## Подготовка к СР №3 (11.03.2027)

СР №3 — полный цикл ML на новых данных. Ты должен уметь:

- [ ] поставить задачу: X, y, тип задачи, утечки (урок 17);
- [ ] разделить данные (`train_test_split`, `stratify` для классификации);
- [ ] обучить модель: дерево, kNN (с масштабированием) или линейную регрессию;
- [ ] сравнить с baseline (`DummyClassifier` / `DummyRegressor`);
- [ ] посчитать метрики: accuracy, матрица ошибок, precision, recall / MAE;
- [ ] заметить переобучение по train и test;
- [ ] написать ограничения модели.

## Самопроверка

1. Чем переобучение отличается от недообучения? Как их узнать по цифрам?
2. Почему нельзя выбирать глубину дерева по test?
3. Зачем нужна validation?
4. Почему разница в 1% на одном разбиении ничего не доказывает?
5. Назови два способа уменьшить переобучение.

**Следующий урок (04.03.2027):** этика ИИ — смещение в данных и ответственность инженера.